# FSRCNN GPU Evaluation — smoke test and full run

This notebook evaluates the verified pretrained FSRCNN(56,12,4) x2, x3, and x4 checkpoints using the same protocol as IMDN. The first evaluation cell runs one Set5 image at every scale and saves those records directly into the resumable full-run checkpoints, so the smoke test is not repeated. The published checkpoints were trained on the 91-image dataset; every result records that fact.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Select a GPU runtime and reconnect.')
DEVICE = torch.device('cuda')
DATASETS = ('Set5', 'Set14', 'BSD100', 'Urban100')
SCALES = (2, 3, 4)
WARMUP_RUNS = 3
TIMED_RUNS = 10
SAVE_RECONSTRUCTIONS = True

print('GPU:', torch.cuda.get_device_name(DEVICE))
print('PyTorch:', torch.__version__)
print('Warm-ups/timed runs per image:', WARMUP_RUNS, '/', TIMED_RUNS)

In [ ]:
from app.deep_learning.checkpoints import (
    FSRCNN_CHECKPOINTS,
    download_pretrained_fsrcnn_checkpoint,
)

DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CHECKPOINT_ROOT = DATA_ROOT / 'checkpoints'
OUTPUT_ROOT = DATA_ROOT / 'results' / 'phase3' / 'fsrcnn' / 'full'
METRICS_ROOT = OUTPUT_ROOT / 'metrics'
SR_ROOT = OUTPUT_ROOT / 'reconstructions'
METRICS_ROOT.mkdir(parents=True, exist_ok=True)

checkpoint_paths = {
    scale: download_pretrained_fsrcnn_checkpoint(CHECKPOINT_ROOT, scale)
    for scale in SCALES
}
for scale in SCALES:
    provenance = FSRCNN_CHECKPOINTS[scale]
    print(
        f'FSRCNN x{scale} verified: {checkpoint_paths[scale]} | '
        f'training={provenance.training_dataset}'
    )

In [ ]:
from app.evaluation.data_validation import validate_prepared_dataset

validations = {}
for dataset in DATASETS:
    for scale in SCALES:
        validation = validate_prepared_dataset(dataset, scale, DATA_ROOT)
        validations[(dataset, scale)] = validation
        print(f'VALID: {dataset} x{scale} — {validation.image_count} pairs')

expected_total = sum(
    validations[(dataset, scale)].image_count
    for dataset in DATASETS
    for scale in SCALES
)
if expected_total != 657:
    raise RuntimeError(f'Expected 657 dataset-scale evaluations; found {expected_total}.')
print('Total expected image evaluations:', expected_total)

In [ ]:
from app.deep_learning.fsrcnn import load_pretrained_fsrcnn
from app.evaluation.experiment import read_results_csv, write_results_csv
from app.evaluation.fsrcnn import FSRCNNEvaluationConfig, evaluate_fsrcnn_image
from app.evaluation.images import pair_image_paths

print('Running one non-duplicated Set5 smoke evaluation per scale...')
for scale in SCALES:
    validation = validations[('Set5', scale)]
    hr_path, lr_path = pair_image_paths(
        validation.hr_directory, validation.lr_directory
    )[0]
    metrics_path = METRICS_ROOT / f'Set5_x{scale}_fsrcnn_gpu.csv'
    existing = read_results_csv(metrics_path)
    if any(record['image'] == hr_path.name for record in existing):
        print(f'FSRCNN x{scale} smoke already complete: {hr_path.name}')
        continue
    model = load_pretrained_fsrcnn(checkpoint_paths[scale], scale, DEVICE)
    config = FSRCNNEvaluationConfig('Set5', scale, WARMUP_RUNS, TIMED_RUNS)
    record = evaluate_fsrcnn_image(
        hr_path, lr_path, model, config, DEVICE,
        sr_output_dir=SR_ROOT / 'Set5' / f'x{scale}',
    )
    if record['parameter_count'] != 12_809:
        raise RuntimeError(f"Unexpected FSRCNN parameter count: {record['parameter_count']}")
    if existing and set(record) != set(existing[0]):
        raise ValueError(f'Resume schema changed for {metrics_path}.')
    existing.append(record)
    write_results_csv(existing, metrics_path, overwrite=True)
    print(
        f"PASS x{scale}: {record['image']} | PSNR-Y {record['psnr_y']:.3f} | "
        f"SSIM-Y {record['ssim_y']:.4f} | GPU {record['latency_mean_ms']:.2f} ms | "
        f"parameters {record['parameter_count']:,}"
    )
    del model
    torch.cuda.empty_cache()

In [ ]:
REQUIRED_RESUME_FIELDS = {
    'dataset', 'image', 'scale', 'method', 'checkpoint_sha256',
    'checkpoint_training_dataset', 'timing_device', 'timing_scope',
    'colour_policy', 'psnr_y', 'ssim_y', 'psnr_rgb', 'ssim_rgb',
    'latency_mean_ms', 'latency_median_ms',
}

def validated_resume_records(csv_path, dataset, scale, expected_names):
    records = read_results_csv(csv_path)
    if not records:
        return []
    missing_fields = REQUIRED_RESUME_FIELDS - set(records[0])
    if missing_fields:
        raise ValueError(f'{csv_path} has an old/incomplete schema: {sorted(missing_fields)}')
    seen = set()
    for record in records:
        if record['dataset'] != dataset or record['scale'] != f'x{scale}':
            raise ValueError(f'{csv_path} contains the wrong dataset or scale.')
        if record['method'] != 'fsrcnn' or record['timing_device'] != 'gpu':
            raise ValueError(f'{csv_path} is not an FSRCNN GPU checkpoint.')
        if record['checkpoint_sha256'] != FSRCNN_CHECKPOINTS[scale].sha256:
            raise ValueError(f'{csv_path} used a different checkpoint.')
        if record['checkpoint_training_dataset'] != '91-image':
            raise ValueError(f'{csv_path} has incorrect training provenance.')
        if record['image'] not in expected_names or record['image'] in seen:
            raise ValueError(f"Unexpected or duplicate image in {csv_path}: {record['image']}")
        seen.add(record['image'])
    return records

In [ ]:
all_records = []
for scale in SCALES:
    model = load_pretrained_fsrcnn(checkpoint_paths[scale], scale, DEVICE)
    print(f'\nLoaded FSRCNN x{scale}')
    for dataset in DATASETS:
        validation = validations[(dataset, scale)]
        pairs = pair_image_paths(validation.hr_directory, validation.lr_directory)
        expected_names = {hr_path.name for hr_path, _ in pairs}
        metrics_path = METRICS_ROOT / f'{dataset}_x{scale}_fsrcnn_gpu.csv'
        records = validated_resume_records(metrics_path, dataset, scale, expected_names)
        completed_names = {record['image'] for record in records}
        print(f'{dataset} x{scale}: {len(completed_names)}/{len(pairs)} complete')
        config = FSRCNNEvaluationConfig(
            dataset, scale, WARMUP_RUNS, TIMED_RUNS
        )
        sr_output_dir = SR_ROOT / dataset / f'x{scale}' if SAVE_RECONSTRUCTIONS else None
        for index, (hr_path, lr_path) in enumerate(pairs, start=1):
            if hr_path.name in completed_names:
                continue
            record = evaluate_fsrcnn_image(
                hr_path, lr_path, model, config, DEVICE,
                sr_output_dir=sr_output_dir,
            )
            if records and set(record) != set(records[0]):
                raise ValueError(f'Resume schema changed for {metrics_path}.')
            records.append(record)
            write_results_csv(records, metrics_path, overwrite=True)
            completed_names.add(hr_path.name)
            print(
                f'  [{index:03d}/{len(pairs):03d}] {hr_path.name} — '
                f"PSNR-Y {record['psnr_y']:.3f}, GPU {record['latency_mean_ms']:.2f} ms"
            )
        if len(records) != len(pairs):
            raise RuntimeError(f'{dataset} x{scale}: {len(records)}/{len(pairs)} records.')
        all_records.extend(records)
        print(f'COMPLETE: {dataset} x{scale} -> {metrics_path}')
    del model
    torch.cuda.empty_cache()

if len(all_records) != expected_total:
    raise RuntimeError(f'Expected {expected_total} records; found {len(all_records)}.')
print(f'\nAll {len(all_records)} FSRCNN image evaluations are complete.')

In [ ]:
from app.evaluation.reporting import summarize_deep_learning_results

dataset_order = {name: index for index, name in enumerate(DATASETS)}
summaries = summarize_deep_learning_results(all_records)
summaries.sort(key=lambda row: (dataset_order[row['dataset']], int(row['scale'][1:])))
all_metrics_path = METRICS_ROOT / 'fsrcnn_all_gpu.csv'
summary_path = METRICS_ROOT / 'fsrcnn_summary_gpu.csv'
write_results_csv(all_records, all_metrics_path, overwrite=True)
write_results_csv(summaries, summary_path, overwrite=True)

print('\nFSRCNN SUMMARY (91-image pretrained checkpoints)')
for row in summaries:
    print(
        f"{row['dataset']:8s} {row['scale']} | PSNR-Y {row['psnr_y']:.3f} | "
        f"SSIM-Y {row['ssim_y']:.4f} | GPU {row['latency_mean_ms']:.2f} ms | "
        f"adjusted {row['dimension_adjusted_count']}/{row['image_count']}"
    )
print('\nCombined metrics:', all_metrics_path)
print('Summary:', summary_path)

## Completion test

Pause after the smoke-test cell only if any scale fails, produces a malformed image, or reports implausible quality. Otherwise continue directly. The full run is complete when it reports **657 FSRCNN image evaluations**, prints 12 summary rows, and saves `fsrcnn_all_gpu.csv` plus `fsrcnn_summary_gpu.csv`. Rerunning all cells safely resumes from per-dataset CSV checkpoints.